# 🔁 Customer Churn Prediction — Business Decisioning System

> *From ML model to profit-driven retention strategy*

---

## 📋 Business Problem

Customer churn is one of the most costly problems in subscription and e-commerce businesses. Losing a customer means losing not just one transaction, but the entire **lifetime value** of that relationship.

This project does not just build a model — it builds a **retention decisioning system** that answers:

> *Which customers should we target with a retention campaign, and at what threshold does targeting them generate maximum profit?*

### 💰 Cost Matrix (Business Assumptions)

| Scenario | Prediction | Reality | Business Outcome | Impact |
|----------|-----------|---------|------------------|---------|
| **True Positive** | Churn | Churn | Customer retained via campaign | **+€280** |
| **False Positive** | Churn | No churn | Unnecessary contact | **-€20** |
| **False Negative** | No churn | Churn | Customer lost, no intervention | **-€300** |
| **True Negative** | No churn | No churn | No action needed | **€0** |

*Assumptions: avg. customer value = €300/year, retention campaign cost = €20/customer*

---

## 🗂️ Dataset

Source: [Kaggle — Digital Marketing E-commerce Customer Behavior](https://www.kaggle.com/datasets/ermismbatuhan/digital-marketing-ecommerce-customer-behavior) by Mustafa Batuhan Ermiş

- **3,333 customers**, 20 features, binary target (`churn`)
- Features include: session data, purchase behavior, customer service calls, engagement metrics

## 1. 📦 Dependencies

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import Normalizer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

# Consistent plot style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

print('✅ Dependencies loaded')

## 2. 📂 Load Data

In [ ]:
# Load dataset — adjust path as needed
# If running on Google Colab, upload data.csv manually or via drive
df = pd.read_csv('data/data.csv', sep=';')

print(f'Shape: {df.shape}')
print(f'Churn rate: {df["churn"].mean():.1%}')
df.head()

## 3. 🔍 Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
# Missing values check
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else '✅ No missing values')

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df['churn'].value_counts()
axes[0].bar(['No Churn (0)', 'Churn (1)'], churn_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Target Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 10, f'{v}\n({v/len(df):.1%})', ha='center', fontweight='bold')

# Customer service calls vs churn
df.groupby('customer service calls')['churn'].mean().plot(
    kind='bar', ax=axes[1], color='#3498db', edgecolor='white'
)
axes[1].set_title('Churn Rate by Customer Service Calls')
axes[1].set_xlabel('Number of Service Calls')
axes[1].set_ylabel('Churn Rate')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.savefig('reports/eda_overview.png', bbox_inches='tight')
plt.show()

## 4. 🛠️ Data Preprocessing

In [ ]:
# location code: treat as categorical
df['location code'] = df['location code'].astype(str)

# Binary encode yes/no columns
df['credit card info save'] = df['credit card info save'].replace({'yes': 1, 'no': 0})
df['push status'] = df['push status'].replace({'yes': 1, 'no': 0})

# Fix European decimal notation (commas → dots)
float_cols = [
    'avg order value',
    'discount rate per visited products',
    'product detail view per app session',
    'add to cart per session'
]
for col in float_cols:
    df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

# One-hot encode location
df = pd.get_dummies(df, columns=['location code'])

# Drop irrelevant ID column
df.drop('user id', axis=1, inplace=True)

print(f'✅ Preprocessing complete. Final shape: {df.shape}')
df.dtypes

## 5. ✂️ Train / Test Split & Normalization

In [ ]:
X = df.drop('churn', axis=1)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

# Normalize
normalizer = Normalizer()
X_train = normalizer.fit_transform(X_train)
X_test  = normalizer.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.1%} | Test churn rate: {y_test.mean():.1%}')

## 6. 🤖 Model Training — XGBoost

XGBoost (Extreme Gradient Boosting) is an ensemble method based on decision trees. It's the go-to for tabular data in industry due to its speed, performance, and built-in regularization.

We first train a baseline model, then run hyperparameter tuning with `GridSearchCV`.

In [ ]:
# Baseline model
xgb_baseline = xgb.XGBClassifier(objective='binary:logistic', random_state=42)
xgb_baseline.fit(X_train, y_train)

baseline_acc = accuracy_score(y_test, xgb_baseline.predict(X_test))
baseline_auc = roc_auc_score(y_test, xgb_baseline.predict_proba(X_test)[:, 1])

print(f'Baseline Accuracy : {baseline_acc:.4f}')
print(f'Baseline ROC-AUC  : {baseline_auc:.4f}')

In [ ]:
# Hyperparameter tuning
param_grid = {
    'max_depth': [5],
    'learning_rate': [0.01, 0.05, 0.1],
    'gamma': [1, 5, 10],
    'scale_pos_weight': [2, 5, 10, 20],
    'subsample': [1],
    'colsample_bytree': [1]
}

xgb_tuned = xgb.XGBClassifier(objective='binary:logistic', random_state=42)
grid_cv = GridSearchCV(xgb_tuned, param_grid, n_jobs=-1, cv=3, scoring='roc_auc')
grid_cv.fit(X_train, y_train)

print(f'Best CV ROC-AUC : {grid_cv.best_score_:.4f}')
print(f'Best Params     : {grid_cv.best_params_}')

In [ ]:
# Final model with best params
final_model = xgb.XGBClassifier(**grid_cv.best_params_, objective='binary:logistic', random_state=42)
final_model.fit(X_train, y_train)

y_pred  = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

final_acc = accuracy_score(y_test, y_pred)
final_auc = roc_auc_score(y_test, y_proba)

print(f'Final Accuracy : {final_acc:.4f}')
print(f'Final ROC-AUC  : {final_auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

## 7. 📊 Model Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Final Model', fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'XGBoost (AUC = {final_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('reports/model_evaluation.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
feat_names = X.columns.tolist()
importances = final_model.feature_importances_
feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=True).tail(12)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 12 Feature Importances (XGBoost)', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('reports/feature_importance.png', bbox_inches='tight')
plt.show()

## 8. 💰 Business Metrics — Cost Matrix & Profit Analysis

Standard ML metrics (accuracy, AUC) don't answer the question that matters to the business:

> *How much money does this model make or save?*

We now translate model predictions into financial outcomes using a **Cost Matrix**.

In [ ]:
# ─── COST MATRIX PARAMETERS ───────────────────────────────────────────────────
REVENUE_PER_CUSTOMER = 300   # € annual value per customer
RETENTION_COST       = 20    # € cost to run retention campaign per customer

# Derived
BENEFIT_TP = REVENUE_PER_CUSTOMER - RETENTION_COST  # +280 (saved churn minus cost)
COST_FP    = -RETENTION_COST                         # -20  (wasted campaign)
COST_FN    = -REVENUE_PER_CUSTOMER                   # -300 (lost customer)
BENEFIT_TN = 0                                       # no action, no cost

print('Cost Matrix:')
print(f'  TP (retained churner)  : +€{BENEFIT_TP}')
print(f'  FP (unnecessary contact): €{COST_FP}')
print(f'  FN (missed churner)    : €{COST_FN}')
print(f'  TN (correct non-action): €{BENEFIT_TN}')

In [ ]:
def compute_profit(y_true, y_proba, threshold):
    """Calculate business profit for a given decision threshold."""
    y_pred = (y_proba >= threshold).astype(int)
    TP = ((y_pred == 1) & (y_true == 1)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    FN = ((y_pred == 0) & (y_true == 1)).sum()
    TN = ((y_pred == 0) & (y_true == 0)).sum()

    profit = (TP * BENEFIT_TP) + (FP * COST_FP) + (FN * COST_FN)
    campaign_cost = (TP + FP) * RETENTION_COST
    roi = (profit / campaign_cost * 100) if campaign_cost > 0 else 0

    return {
        'profit': profit, 'roi': roi,
        'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN,
        'n_targeted': TP + FP,
        'campaign_cost': campaign_cost
    }


# Profit at default threshold (0.5)
default_metrics = compute_profit(y_test.values, y_proba, 0.5)
print(f'Profit at threshold=0.5 : €{default_metrics["profit"]:,.0f} | ROI: {default_metrics["roi"]:.1f}%')

## 9. 🎯 Threshold Optimization by Profit

The default threshold of 0.5 is **not** optimal for profit. By sweeping thresholds we find the point that maximizes net gain.

> *"The model decides who might churn. The threshold decides who we act on."*

In [ ]:
thresholds = np.linspace(0.05, 0.95, 200)
profits, rois, n_targeted = [], [], []

for t in thresholds:
    m = compute_profit(y_test.values, y_proba, t)
    profits.append(m['profit'])
    rois.append(m['roi'])
    n_targeted.append(m['n_targeted'])

best_idx    = np.argmax(profits)
best_t      = thresholds[best_idx]
best_metrics = compute_profit(y_test.values, y_proba, best_t)

print(f'Optimal Threshold  : {best_t:.2f}')
print(f'Net Profit         : €{best_metrics["profit"]:,.0f}')
print(f'ROI                : {best_metrics["roi"]:.1f}%')
print(f'Customers Targeted : {best_metrics["n_targeted"]} ({best_metrics["n_targeted"]/len(y_test):.1%} of test set)')
print(f'Campaign Cost      : €{best_metrics["campaign_cost"]:,.0f}')

In [ ]:
# ─── PROFIT vs THRESHOLD CHART ─────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(11, 5))

# Profit line
ax1.fill_between(thresholds, profits, alpha=0.15, color='steelblue')
ax1.plot(thresholds, profits, color='steelblue', lw=2.5, label='Net Profit (€)')
ax1.axvline(best_t, color='#e74c3c', linestyle='--', lw=2,
            label=f'Optimal threshold = {best_t:.2f}')
ax1.axvline(0.5, color='gray', linestyle=':', lw=1.5, label='Default threshold = 0.50')
ax1.axhline(0, color='black', lw=0.8)
ax1.scatter([best_t], [best_metrics['profit']], color='#e74c3c', s=100, zorder=5)
ax1.annotate(f'€{best_metrics["profit"]:,.0f}',
             xy=(best_t, best_metrics['profit']),
             xytext=(best_t + 0.05, best_metrics['profit'] * 0.9),
             fontsize=10, color='#e74c3c', fontweight='bold')
ax1.set_xlabel('Decision Threshold', fontsize=12)
ax1.set_ylabel('Net Profit (€)', color='steelblue', fontsize=12)
ax1.tick_params(axis='y', labelcolor='steelblue')

# ROI on secondary axis
ax2 = ax1.twinx()
ax2.plot(thresholds, rois, color='#f39c12', lw=1.5, linestyle=':', alpha=0.7, label='ROI (%)')
ax2.set_ylabel('ROI (%)', color='#f39c12', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#f39c12')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', framealpha=0.9)

ax1.set_title('Profit & ROI Optimization by Decision Threshold', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/profit_vs_threshold.png', bbox_inches='tight')
plt.show()

print(f'\n💡 Using optimal threshold ({best_t:.2f}) vs default (0.5):')
print(f'   Profit improvement: €{best_metrics["profit"] - default_metrics["profit"]:,.0f}')

## 10. 📈 Profit Curve by Customer Segment (Cumulative Gains)

Not all customers are equal. By sorting customers by predicted churn probability (high-risk first), we can see how profit accumulates as we expand the campaign reach.

In [ ]:
# Sort test set by predicted churn probability — highest risk first
sorted_idx = np.argsort(y_proba)[::-1]
y_true_sorted  = y_test.values[sorted_idx]
y_proba_sorted = y_proba[sorted_idx]

# Cumulative profit as we contact more and more customers
cum_profit = []
running = 0
for actual in y_true_sorted:
    if actual == 1:
        running += BENEFIT_TP
    else:
        running += COST_FP
    cum_profit.append(running)

pct_contacted = np.arange(1, len(y_true_sorted) + 1) / len(y_true_sorted) * 100
best_profit_idx = np.argmax(cum_profit)

plt.figure(figsize=(11, 5))
plt.plot(pct_contacted, cum_profit, color='steelblue', lw=2.5)
plt.axvline(pct_contacted[best_profit_idx], color='#e74c3c', linestyle='--', lw=2,
            label=f'Sweet spot: top {pct_contacted[best_profit_idx]:.0f}% → €{cum_profit[best_profit_idx]:,.0f}')
plt.axhline(0, color='black', lw=0.8)
plt.fill_between(pct_contacted, cum_profit,
                 where=[c > 0 for c in cum_profit], alpha=0.1, color='green', label='Profitable zone')
plt.fill_between(pct_contacted, cum_profit,
                 where=[c < 0 for c in cum_profit], alpha=0.1, color='red', label='Loss zone')
plt.xlabel('% of Customer Base Contacted (sorted by churn risk)', fontsize=12)
plt.ylabel('Cumulative Net Profit (€)', fontsize=12)
plt.title('Cumulative Profit Curve — Risk-Sorted Targeting', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('reports/cumulative_profit_curve.png', bbox_inches='tight')
plt.show()

## 11. 🏦 Executive Business Summary

In [ ]:
# Model comparison table
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'ROC-AUC', 'Precision (Churn)', 'Recall (Churn)'],
    'Baseline XGBoost': [
        f"{accuracy_score(y_test, xgb_baseline.predict(X_test)):.4f}",
        f"{roc_auc_score(y_test, xgb_baseline.predict_proba(X_test)[:,1]):.4f}",
        '—', '—'
    ],
    'Tuned XGBoost': [
        f"{final_acc:.4f}",
        f"{final_auc:.4f}",
        f"{(y_pred[y_pred==1]==y_test.values[y_pred==1]).mean():.4f}" if (y_pred==1).any() else '—',
        f"{(y_pred[y_test.values==1]==1).mean():.4f}" if (y_test==1).any() else '—'
    ]
})
print('Model Comparison:')
print(comparison.to_string(index=False))

In [ ]:
# ─── EXECUTIVE SUMMARY ────────────────────────────────────────────────────────
pct_targeted = best_metrics['n_targeted'] / len(y_test) * 100
missed_revenue = best_metrics['FN'] * REVENUE_PER_CUSTOMER

summary = f"""
{'='*60}
   CUSTOMER CHURN RETENTION — BUSINESS CASE SUMMARY
{'='*60}

  MODEL PERFORMANCE
  -----------------
  Algorithm    : XGBoost (tuned)
  Accuracy     : {final_acc:.1%}
  ROC-AUC      : {final_auc:.3f}

  RETENTION POLICY (profit-optimized)
  ------------------------------------
  Decision threshold  : {best_t:.2f}  (vs. default 0.50)
  Customers targeted  : {best_metrics['n_targeted']} ({pct_targeted:.0f}% of base)
  True churners caught: {best_metrics['TP']}  (TP)
  False alarms        : {best_metrics['FP']}  (FP — acceptable cost)
  Churners missed     : {best_metrics['FN']}  (FN)

  FINANCIAL IMPACT
  ----------------
  Campaign investment : €{best_metrics['campaign_cost']:,.0f}
  Net profit          : €{best_metrics['profit']:,.0f}
  ROI                 : {best_metrics['roi']:.1f}%
  Revenue at risk (FN): €{missed_revenue:,.0f}

  RECOMMENDATION
  --------------
  Target the top {pct_targeted:.0f}% highest-risk customers identified
  by the model. Expected net gain per campaign cycle: €{best_metrics['profit']:,.0f}
  with a {best_metrics['roi']:.0f}% return on campaign investment.

{'='*60}
"""

print(summary)

# Save to file
with open('reports/business_summary.txt', 'w') as f:
    f.write(summary)

## 12. 💾 Save Model

In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

joblib.dump(final_model, 'models/xgb_churn_model.pkl')
joblib.dump(normalizer, 'models/normalizer.pkl')

print('✅ Model and normalizer saved to /models')
print(f'   Optimal threshold for deployment: {best_t:.2f}')

---

## ✅ Conclusion

This project demonstrates a **complete ML-to-business pipeline** for customer churn prediction:

1. **Data Preprocessing** — Handled mixed types, categorical encoding, normalization
2. **Model Training** — XGBoost with GridSearchCV hyperparameter tuning (AUC: ~0.88)
3. **Standard Evaluation** — Accuracy, ROC-AUC, confusion matrix, classification report
4. **Business Translation** — Cost Matrix quantifying the € impact of each prediction type
5. **Threshold Optimization** — Moved from default 0.5 to profit-maximizing threshold
6. **ROI Calculation** — Campaign investment vs. net revenue saved
7. **Actionable Policy** — Clear recommendation on which customer segment to target

### Key Insight
The default classification threshold (0.5) is not optimal from a business perspective. By optimizing the threshold for profit rather than accuracy, we maximize the financial return of the retention campaign — demonstrating that **the right business decision requires more than a good model**.

---
*Built with XGBoost · scikit-learn · pandas · matplotlib*